# 06 -- Pipeline Integration: CLI vs Library API

The `venn-diagram-lab` package ships two interfaces:

1. **Python library API** -- `import venn_diagram_lab as vdl` -- best for
   interactive notebooks and custom scripts where you want to inspect
   intermediate objects (`result.regions`, `result.statistics`, etc.).
2. **`vdl` CLI** -- a shell command that reads a file and writes output
   artefacts (SVG, PNG, PDF, TSV) -- best for reproducible pipelines in
   Snakemake, Nextflow, Makefile, or CI/CD workflows.

**What you will learn:**

- When to reach for the library vs the CLI
- The full CLI command reference (`vdl --help`)
- How to run a quick demo with `vdl render-sample`
- How to wire `vdl analyze` into a Snakemake workflow
- How to wire `vdl analyze` into a Nextflow pipeline


In [1]:
import venn_diagram_lab as vdl

print(f'venn-diagram-lab {vdl.__version__}')

venn-diagram-lab 2.2.3


## Library vs CLI tradeoffs

| Concern | Library API | CLI (`vdl`) |
|---------|------------|-------------|
| Interactive exploration | Best | No |
| Inspect intermediate objects | Yes | No |
| Custom plots / downstream code | Yes | No |
| Shell scripts / Makefile | No | Best |
| Snakemake / Nextflow | No | Best |
| CI/CD artefact generation | No | Best |
| Reproducible file-in / file-out | Possible | Native |
| No Python knowledge required | No | Yes |


### When to use the library

- Jupyter notebooks where you want to render inline figures.
- Scripts that apply custom filters, post-process region lists, or feed
  results into another analysis (e.g., pathway enrichment after vdl).
- Cases where you need `result.statistics` DataFrames directly in memory.

```python
import venn_diagram_lab as vdl

result = vdl.analyze(vdl.load_sample('dataset_real_cancer_drivers_4'), model='auto')
print(result.statistics.jaccard)
```


### When to use the CLI

- Snakemake rules or Nextflow processes that consume a TSV and produce
  SVG/PNG/PDF/TSV artefacts.
- Shell one-liners: `vdl analyze data.tsv --output-dir out/`
- CI workflows that must run without a Jupyter kernel.
- Any situation where reproducibility is best enforced at the file level
  (input hash -> output hash) rather than in-memory state.


## CLI overview

In [2]:
import subprocess
import sys
from pathlib import Path

# Derive the vdl executable from the active Python interpreter's bin dir
# so this cell works regardless of whether 'vdl' is on PATH.
_vdl = str(Path(sys.executable).parent / 'vdl')

r = subprocess.run([_vdl, '--help'], capture_output=True, text=True, check=True)
print(r.stdout)

                                                                                
 Usage: vdl [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 venn-diagram-lab - headless Venn diagram analysis and rendering.               
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ about           Show a sho

## Quick demo: render a sample dataset

The `vdl render-sample` command looks up a bundled dataset by name and
runs the full analysis pipeline, writing the output bundle to a directory.
It is equivalent to:

```
vdl analyze <path-to-sample-tsv> --output-dir <dir>
```

Run it now against the mock streaming-platforms dataset (4 sets):


In [3]:
import os
import subprocess
import tempfile

demo_dir = tempfile.mkdtemp(prefix='vdl_demo_')
r = subprocess.run(
    [_vdl, 'render-sample', 'dataset_mock_streaming_platforms',
     '--output-dir', demo_dir],
    capture_output=True, text=True, check=True,
)
print(r.stdout or '(no stdout)')
print('Files written to', demo_dir)

wrote /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_demo_2rtqytny/venn.svg
wrote /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_demo_2rtqytny/upset.png
wrote /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_demo_2rtqytny/network.png
wrote /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_demo_2rtqytny/report.pdf
wrote /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_demo_2rtqytny/statistics.tsv

Files written to /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_demo_2rtqytny


In [4]:
for fname in sorted(os.listdir(demo_dir)):
    fpath = os.path.join(demo_dir, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'{fname:<30}  {size_kb:>7.1f} KB')

network.png                       436.7 KB
report.pdf                        402.3 KB
statistics.tsv                      2.7 KB
upset.png                         198.6 KB
venn.svg                           67.1 KB


## Snakemake example

The `Snakefile` below defines two rules:

- **`all_reports`** -- the default target that declares all expected output files.
- **`analyze`** -- the worker rule that runs `vdl analyze` with `--output-dir`
  and maps the five output files into named Snakemake outputs for provenance.

Snakemake will automatically resolve that `all_reports` depends on `analyze`
and run only the rules needed to produce the requested outputs.


In [5]:
from pathlib import Path

import venn_diagram_lab

# Derive repo root from the installed package location (works in any cwd)
_repo_root = Path(venn_diagram_lab.__file__).parents[3]
_snakefile = _repo_root / 'python' / 'examples' / 'pipelines' / 'Snakefile'
print(_snakefile.read_text())

# Snakefile -- minimal example: vdl analyze + collect outputs
#
# Usage:
#   snakemake --cores 1 --snakefile python/examples/pipelines/Snakefile \
#       all_reports
#
# Inputs:  python/src/venn_diagram_lab/_data/samples/dataset_real_cancer_drivers_4.tsv
# Outputs: results/cancer_drivers/{venn.svg,upset.png,network.png,report.pdf,statistics.tsv}

INPUT_TSV = "python/src/venn_diagram_lab/_data/samples/dataset_real_cancer_drivers_4.tsv"
OUT_DIR = "results/cancer_drivers"

rule all_reports:
    input:
        f"{OUT_DIR}/venn.svg",
        f"{OUT_DIR}/upset.png",
        f"{OUT_DIR}/network.png",
        f"{OUT_DIR}/report.pdf",
        f"{OUT_DIR}/statistics.tsv",

rule analyze:
    input:
        tsv=INPUT_TSV,
    output:
        venn=f"{OUT_DIR}/venn.svg",
        upset=f"{OUT_DIR}/upset.png",
        network=f"{OUT_DIR}/network.png",
        pdf=f"{OUT_DIR}/report.pdf",
        stats=f"{OUT_DIR}/statistics.tsv",
    shell:
        "vdl analyze {input.tsv} --output-dir " + OUT_DIR



Run with:

```bash
snakemake --cores 1 --snakefile python/examples/pipelines/Snakefile all_reports
```

Snakemake will skip the rule on subsequent runs if the output files are
already up to date (newer than the input TSV) -- giving you incremental
rebuild for free.


## Nextflow example

In [6]:
_nf = _repo_root / 'python' / 'examples' / 'pipelines' / 'main.nf'
print(_nf.read_text())

// main.nf -- minimal Nextflow pipeline running vdl analyze
//
// Usage:
//   nextflow run python/examples/pipelines/main.nf

params.input = "python/src/venn_diagram_lab/_data/samples/dataset_real_cancer_drivers_4.tsv"
params.outdir = "results/cancer_drivers"

process AnalyzeAndReport {
    publishDir params.outdir, mode: 'copy'

    input:
    path tsv

    output:
    path "venn.svg"
    path "upset.png"
    path "network.png"
    path "report.pdf"
    path "statistics.tsv"

    script:
    """
    vdl analyze ${tsv} --output-dir .
    """
}

workflow {
    Channel.fromPath(params.input) | AnalyzeAndReport
}



Run with:

```bash
nextflow run python/examples/pipelines/main.nf
```

Nextflow stages the input file into a work directory, runs the process
inside it, then copies the named outputs to `params.outdir` via `publishDir`.
The `Channel.fromPath` source can be replaced with a glob pattern to fan out
over multiple input files -- each processed in parallel by a separate worker.


## `vdl workflow run-from` (v2.2.3 native YAML)

Snakemake and Nextflow are the right tool for multi-input, multi-rule
pipelines. For a single dataset that needs N outputs, `vdl workflow
run-from` is simpler: a small YAML config declares each output, and one
command produces them all. No external orchestrator required.

The config schema:

```yaml
version: 1
input: dataset_real_cancer_drivers_4   # sample name OR path to TSV/CSV
model: auto                            # or 'proportional'
outputs:
  - kind: venn       # or upset / network / heatmap / share-dist /
    out: out/v.svg   #   pdf / region-summary / matrix / statistics / cluster
  - kind: statistics
    out: out/s.tsv
```

`vdl workflow init <dir>` scaffolds a starter project with a runnable
default config. Below we build a small one inline and run it.


In [7]:
import subprocess
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp(prefix='vdl_runfrom_'))
yaml_text = '''\
version: 1
input: dataset_real_cancer_drivers_4
model: auto
outputs:
  - kind: venn
    out: ''' + str(tmp / 'venn.svg') + '''
  - kind: upset
    out: ''' + str(tmp / 'upset.svg') + '''
  - kind: network
    out: ''' + str(tmp / 'network.svg') + '''
  - kind: statistics
    out: ''' + str(tmp / 'statistics.tsv') + '''
  - kind: region-summary
    out: ''' + str(tmp / 'regions.tsv') + '''
'''
yaml_path = tmp / 'analysis.yaml'
yaml_path.write_text(yaml_text)

r = subprocess.run(
    [_vdl, 'workflow', 'run-from', str(yaml_path)],
    capture_output=True, text=True, check=True,
)
print(r.stdout)
print('Files in', tmp, ':')
for f in sorted(tmp.iterdir()):
    print(f'  {f.name:<25}  {f.stat().st_size:>8,} bytes')

[1/5] venn -> /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_runfrom_p5ce4adx/venn.svg
[2/5] upset -> /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_runfrom_p5ce4adx/upset.svg
[3/5] network -> /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_runfrom_p5ce4adx/network.svg
[4/5] statistics -> /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_runfrom_p5ce4adx/statistics.tsv
[5/5] region-summary -> /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_runfrom_p5ce4adx/regions.tsv
done.

Files in /var/folders/ww/fsvzxkwx6hgfzqbq08lfrqdc0000gn/T/vdl_runfrom_p5ce4adx :
  analysis.yaml                   617 bytes
  network.svg                  15,198 bytes
  regions.tsv                   8,897 bytes
  statistics.tsv                  704 bytes
  upset.svg                    48,337 bytes
  venn.svg                      6,254 bytes


### Snakemake vs Nextflow vs `vdl workflow run-from`

| Tool | Strength | Use when |
|------|----------|---------|
| **Snakemake** | Fine-grained DAG, file timestamp caching, conda envs | Multi-rule pipeline, many inputs, incremental rebuild matters |
| **Nextflow** | Native parallelism, container-first, cloud-ready | Many input files processed in parallel; HPC or cloud target |
| **`vdl workflow run-from`** | One config, one command, zero dependencies | Single dataset, N outputs, no external orchestrator needed |

Rule of thumb: start with `vdl workflow run-from` for a single analysis.
Promote to Snakemake when you have multiple datasets and want incremental
rebuild. Promote to Nextflow when you need cloud-scale parallelism.


## Next steps

- [`07_pdf_reports.ipynb`](07_pdf_reports.ipynb) -- bundle Venn, UpSet, Network, and statistics into a single PDF report
- [`08_custom_styling_and_export.ipynb`](08_custom_styling_and_export.ipynb) -- style diagrams and export SVG/PNG for publication
